# AIML Lab Assignment #11 — RNN for Time-Series Forecasting
**Course:** CSET301 | **Dataset:** Jena Climate Dataset | **Semester:** 4th Even 2026

---

## What will we do?
We will build a **Recurrent Neural Network (RNN)** to **predict future temperature** from past weather measurements.

```
Past 24 hours of weather data  →  RNN  →  Next hour's temperature
[temp, pressure, humidity ...]       →  Predict: 15.3°C
```

### Steps:
1. **Load Dataset** — Jena Climate (temperature + 13 sensor readings)
2. **Preprocessing** — Parse timestamps, normalize, create sequences
3. **Build RNN/LSTM Model** — Recurrent layers + Dropout + Dense
4. **Train & Compare** — Train for 20 epochs vs 50 epochs
5. **Save Model** — Save trained model to file
6. **Evaluate** — MAE, RMSE, predicted vs actual plots

---

### What is an RNN?
A **Recurrent Neural Network** has a special ability: it has **memory**.
Unlike a regular neural network that processes one input at a time independently,
an RNN passes information from one time step to the next.

```
Normal NN:   Input → Output  (no memory)

RNN:  Input₁ → Hidden₁ → Output₁
                  ↓
      Input₂ → Hidden₂ → Output₂   ← Hidden₁ is passed here!
                  ↓
      Input₃ → Hidden₃ → Output₃   ← Hidden₂ is passed here!
```

**LSTM** (Long Short-Term Memory) is a special improved RNN that remembers
patterns over longer time periods without forgetting.

### Why RNN for time-series?
Temperature at 3pm depends on temperature at 2pm, 1pm, 12pm...
RNN naturally handles this sequential dependency — regular networks cannot!

---
## Step 0: Import All Libraries

In [ ]:
# ================================================================
# IMPORT ALL LIBRARIES
# ================================================================

# numpy  → number arrays and math
import numpy as np

# pandas → load and handle the CSV dataset as a table
import pandas as pd

# matplotlib → draw all our charts and plots
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# seaborn → prettier heatmaps and correlation plots
import seaborn as sns

# sklearn → data splitting and metrics
from sklearn.preprocessing  import MinMaxScaler
from sklearn.metrics        import mean_absolute_error, mean_squared_error

# tensorflow / keras → build and train the RNN / LSTM
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# os → file system operations (for saving model)
import os
import warnings
warnings.filterwarnings('ignore')

# Fix random seeds so results are reproducible
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("All libraries imported successfully!")

---
## Step 1: Dataset Loading

### About the Jena Climate Dataset
- Recorded at the **Max Planck Institute for Biogeochemistry** in Jena, Germany
- Contains **14 sensor measurements** recorded every **10 minutes** from 2009–2016
- We want to predict **temperature (°C)** from past sensor readings

| Column | Description |
|---|---|
| `Date Time` | Timestamp of measurement |
| `T (degC)` | Temperature in Celsius ← our **target** |
| `p (mbar)` | Air pressure |
| `rh (%)` | Relative humidity |
| `wv (m/s)` | Wind velocity |
| `wd (deg)` | Wind direction |

In [ ]:
# ================================================================
# 1.1  LOAD THE JENA CLIMATE DATASET
# ================================================================
# The dataset is ~82MB — it downloads once and is cached
# If the URL fails, uncomment the fallback synthetic data below

print("Downloading Jena Climate dataset...")
print("(~82MB — may take a minute on first run)")
print()

URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"

try:
    df = pd.read_csv(URL, compression='zip')
    print("Dataset downloaded successfully!")

except Exception as e:
    print(f"Download failed ({e})")
    print("Generating synthetic climate-like dataset instead...")

    # ---- SYNTHETIC FALLBACK ----
    # Simulates realistic temperature data with daily + seasonal cycles
    n = 50000  # ~1 year of 10-minute measurements
    t = np.arange(n)

    # Seasonal cycle (yearly) + daily cycle + noise
    temp = (10                                          # baseline
            + 10 * np.sin(2 * np.pi * t / (6 * 24 * 365))  # seasonal
            + 5  * np.sin(2 * np.pi * t / (6 * 24))         # daily
            + np.random.normal(0, 1, n))                     # noise

    dates = pd.date_range('2009-01-01 00:10:00', periods=n, freq='10min')
    df = pd.DataFrame({
        'Date Time'  : dates.strftime('%d.%m.%Y %H:%M:%S'),
        'p (mbar)'   : 1013 + np.random.normal(0, 5, n),
        'T (degC)'   : temp,
        'Tpot (K)'   : temp + 273.15,
        'Tdew (degC)': temp - 3 + np.random.normal(0, 1, n),
        'rh (%)'     : 70 + np.random.normal(0, 10, n),
        'VPmax (mbar)': 20 + np.random.normal(0, 3, n),
        'VPact (mbar)': 14 + np.random.normal(0, 2, n),
        'VPdef (mbar)': 6  + np.random.normal(0, 1, n),
        'sh (g/kg)'  : 8  + np.random.normal(0, 1, n),
        'H2OC (mmol/mol)': 12 + np.random.normal(0, 1, n),
        'rho (g/m**3)'   : 1200 + np.random.normal(0, 20, n),
        'wv (m/s)'   : 3  + np.abs(np.random.normal(0, 2, n)),
        'max. wv (m/s)': 5 + np.abs(np.random.normal(0, 2, n)),
        'wd (deg)'   : np.random.uniform(0, 360, n),
    })
    print("Synthetic dataset ready!")

print()
print(f"Dataset shape : {df.shape}  → {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns       : {list(df.columns)}")
print()
print("First 3 rows:")
print(df.head(3))

In [ ]:
# ================================================================
# 1.2  INSPECT THE DATASET
# ================================================================

print("===== Dataset Basic Info =====")
print(f"Total rows         : {len(df):,}")
print(f"Total columns      : {df.shape[1]}")
print(f"Missing values     : {df.isnull().sum().sum()}")
print()
print("Statistical Summary:")
print(df.describe().round(2))

In [ ]:
# ================================================================
# 1.3  VISUALIZE RAW TEMPERATURE DATA
# ================================================================
# Plot first 5000 rows (~35 days) to see the seasonal and daily patterns

fig, axes = plt.subplots(3, 1, figsize=(16, 10))
fig.suptitle('Jena Climate Dataset — Raw Sensor Readings', fontsize=13, fontweight='bold')

n_plot = 5000   # plot first 5000 rows for clarity

axes[0].plot(df['T (degC)'].values[:n_plot], color='coral',     linewidth=0.8)
axes[0].set_title('Temperature (°C) — Shows daily cycle', fontweight='bold')
axes[0].set_ylabel('°C'); axes[0].grid(alpha=0.3)

axes[1].plot(df['p (mbar)'].values[:n_plot], color='steelblue', linewidth=0.8)
axes[1].set_title('Air Pressure (mbar)', fontweight='bold')
axes[1].set_ylabel('mbar'); axes[1].grid(alpha=0.3)

axes[2].plot(df['rh (%)'].values[:n_plot], color='green',      linewidth=0.8)
axes[2].set_title('Relative Humidity (%)', fontweight='bold')
axes[2].set_xlabel('Time Step (10-minute intervals)')
axes[2].set_ylabel('%'); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print("Notice the repeating daily cycle in temperature — RNN will learn to predict this!")

---
## Step 2: Data Preprocessing

### What we need to do:
1. **Parse timestamps** → extract hour, day, month features
2. **Select features** → temperature (target) + sensor readings (inputs)
3. **Normalize** → scale all values to [0, 1]
4. **Downsample** → dataset is every 10 min; we use hourly data (every 6 rows)
5. **Create sequences** → sliding window: past 24 hours → next hour temperature
6. **Reshape** → (samples, time_steps, features) for RNN input

In [ ]:
# ================================================================
# 2.1  PARSE TIMESTAMPS — extract hour, day, month
# ================================================================
# The 'Date Time' column is a string like '01.01.2009 00:10:00'
# We need to convert it to actual datetime and extract useful parts

# Convert 'Date Time' column to pandas datetime format
df['Date Time'] = pd.to_datetime(df['Date Time'], dayfirst=True)

# Extract time features from the timestamp
# These help the model understand seasonal and daily patterns
df['hour']  = df['Date Time'].dt.hour    # 0-23 → time of day
df['month'] = df['Date Time'].dt.month   # 1-12 → season
df['day']   = df['Date Time'].dt.day     # 1-31 → day of month

# Set 'Date Time' as the index for easy time-based plotting
df = df.set_index('Date Time')

print("Timestamp parsing done!")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"New columns added: hour, month, day")
print()
print(df[['T (degC)', 'hour', 'month', 'day']].head(8))

In [ ]:
# ================================================================
# 2.2  DOWNSAMPLE — use every 6th row (hourly from 10-minute data)
# ================================================================
# The dataset records every 10 minutes.
# 6 rows × 10 minutes = 60 minutes = 1 hour.
# We use hourly data for simpler and faster training.

df_hourly = df.iloc[::6].copy()   # take every 6th row

print(f"Original rows : {len(df):,}   (every 10 minutes)")
print(f"After downsample: {len(df_hourly):,}   (every 1 hour)")
print(f"Date range    : {df_hourly.index.min()} to {df_hourly.index.max()}")

In [ ]:
# ================================================================
# 2.3  SELECT FEATURES
# ================================================================
# We use all numeric sensor readings as INPUT features
# TARGET = temperature 'T (degC)'

# All feature columns (input to the model)
FEATURE_COLS = [
    'p (mbar)', 'T (degC)', 'Tpot (K)', 'Tdew (degC)',
    'rh (%)', 'VPmax (mbar)', 'VPact (mbar)', 'VPdef (mbar)',
    'sh (g/kg)', 'H2OC (mmol/mol)', 'rho (g/m**3)',
    'wv (m/s)', 'max. wv (m/s)', 'wd (deg)',
    'hour', 'month', 'day'
]

# Filter to only columns that exist in the DataFrame
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_hourly.columns]

# Target column index (position of 'T (degC)' in FEATURE_COLS)
TARGET_COL = FEATURE_COLS.index('T (degC)')

# Extract the feature matrix
data = df_hourly[FEATURE_COLS].values   # shape: (N, n_features)

print(f"Feature columns ({len(FEATURE_COLS)} total):")
for i, col in enumerate(FEATURE_COLS):
    marker = '  ← TARGET' if col == 'T (degC)' else ''
    print(f"  [{i:>2}] {col}{marker}")
print()
print(f"Data matrix shape: {data.shape}  → {data.shape[0]:,} hours × {data.shape[1]} features")

In [ ]:
# ================================================================
# 2.4  NORMALIZE — scale all values to [0, 1]
# ================================================================
# Neural networks train better with small values.
# MinMaxScaler scales each column independently:
#   scaled = (value - min) / (max - min)  → always in [0, 1]
#
# IMPORTANT: Fit scaler on TRAIN data only, then transform all sets
# (to avoid leaking future information into training)

# First, split time-series into 80% train / 20% test
# For time-series, we CANNOT shuffle! We must keep chronological order.
split_idx = int(len(data) * 0.80)   # 80% for training

train_data_raw = data[:split_idx]   # first 80% (earlier in time)
test_data_raw  = data[split_idx:]   # last 20%  (later in time)

print(f"Training rows  : {len(train_data_raw):,}  (first 80% — chronological order!)")
print(f"Test rows      : {len(test_data_raw):,}   (last 20%)")
print()

# Fit scaler ONLY on training data
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_data_raw)   # fit + transform
test_scaled  = scaler.transform(test_data_raw)         # only transform

print(f"Before scaling — Temp range: {train_data_raw[:, TARGET_COL].min():.1f}°C to {train_data_raw[:, TARGET_COL].max():.1f}°C")
print(f"After  scaling — Temp range: {train_scaled[:, TARGET_COL].min():.2f} to {train_scaled[:, TARGET_COL].max():.2f}")

In [ ]:
# ================================================================
# 2.5  CREATE INPUT-OUTPUT SEQUENCES
# ================================================================
# We use a SLIDING WINDOW approach:
#
#  Hours: [1,2,3,...,24] → predict hour 25
#  Hours: [2,3,4,...,25] → predict hour 26
#  Hours: [3,4,5,...,26] → predict hour 27
#  ... and so on
#
# Each input = 24 consecutive hours of all features
# Each output = temperature value at the next hour

TIME_STEPS = 24   # look back 24 hours to predict next hour

def create_sequences(data, target_col, time_steps=24):
    """
    Create sliding window sequences from time-series data.

    For each position i:
      Input  = rows i to i+time_steps   (shape: time_steps × n_features)
      Output = target value at row i+time_steps (single number)

    Returns:
      sequences: shape (n_samples, time_steps, n_features)
      labels   : shape (n_samples,)
    """
    sequences = []   # will hold input windows
    labels    = []   # will hold target temperatures

    for i in range(len(data) - time_steps):
        # Input window: rows i through i+time_steps
        sequences.append(data[i : i + time_steps])

        # Target: temperature at the NEXT time step after the window
        labels.append(data[i + time_steps, target_col])

    return np.array(sequences), np.array(labels)


# Create sequences for training and test sets
X_train, y_train = create_sequences(train_scaled, TARGET_COL, TIME_STEPS)
X_test,  y_test  = create_sequences(test_scaled,  TARGET_COL, TIME_STEPS)

print("===== Sequence Creation Summary =====")
print(f"X_train shape : {X_train.shape}")
print(f"  → {X_train.shape[0]:,} samples × {X_train.shape[1]} time steps × {X_train.shape[2]} features")
print(f"y_train shape : {y_train.shape}  → one temperature value per sample")
print()
print(f"X_test shape  : {X_test.shape}")
print(f"y_test shape  : {y_test.shape}")
print()
print("Each X sample = 24 hours of weather readings (24 × n_features matrix)")
print("Each y value  = temperature at hour 25")

In [ ]:
# ================================================================
# 2.6  VISUALIZE CORRELATION — which features relate to temperature?
# ================================================================

corr = pd.DataFrame(train_data_raw, columns=FEATURE_COLS).corr()

plt.figure(figsize=(13, 10))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, linewidths=0.4, annot_kws={'size': 7}
)
plt.title('Feature Correlation Matrix\n(How much does each feature relate to Temperature?)',
          fontweight='bold')
plt.tight_layout()
plt.show()
print("High |correlation| with T (degC) → that feature is useful for prediction!")

---
## Step 3: Model Construction

We build two models:
- **Simple RNN** — basic recurrent layer
- **LSTM** — improved RNN with longer memory (usually better)

### Architecture:
```
Input: (batch, 24 time_steps, n_features)
         ↓
    LSTM(64)       ← recurrent layer, processes sequence step by step
         ↓
    Dropout(0.2)   ← randomly turn off 20% of neurons (prevent overfitting)
         ↓
    LSTM(32)       ← second recurrent layer
         ↓
    Dropout(0.2)
         ↓
    Dense(16)      ← fully connected layer
         ↓
    Dense(1)       ← single output = predicted temperature
```

In [ ]:
# ================================================================
# 3.1  BUILD SIMPLE RNN MODEL
# ================================================================

N_FEATURES = X_train.shape[2]   # number of input features

def build_simple_rnn(time_steps, n_features):
    """
    Basic SimpleRNN model.
    SimpleRNN has very short memory — struggles with long sequences.
    Good for understanding the concept before using LSTM.
    """
    model = keras.Sequential([
        # Input shape: (time_steps, n_features)
        layers.Input(shape=(time_steps, n_features)),

        # SimpleRNN layer: processes sequence one time step at a time
        # return_sequences=True: pass output at EACH step to next layer
        layers.SimpleRNN(64, activation='tanh', return_sequences=True),
        layers.Dropout(0.2),   # 20% of neurons turned off randomly

        # Second RNN layer — return_sequences=False: only return LAST output
        layers.SimpleRNN(32, activation='tanh', return_sequences=False),
        layers.Dropout(0.2),

        # Dense layers for final prediction
        layers.Dense(16, activation='relu'),

        # Output: single number = next hour's temperature (scaled)
        layers.Dense(1)

    ], name='Simple_RNN')

    return model


rnn_model = build_simple_rnn(TIME_STEPS, N_FEATURES)

# Compile: MSE loss (minimizes squared error) + Adam optimizer
rnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',           # Mean Squared Error — standard for regression
    metrics=['mae']       # also track Mean Absolute Error
)

rnn_model.summary()
print()
print("Input shape : (batch, 24 time_steps, n_features)")
print("Output shape: (batch, 1) — one temperature prediction per sequence")

In [ ]:
# ================================================================
# 3.2  BUILD LSTM MODEL
# ================================================================
# LSTM = Long Short-Term Memory
# Has special 'gates' that control what to remember and what to forget
# Much better than SimpleRNN for long sequences like 24-hour weather!

def build_lstm_model(time_steps, n_features):
    """
    LSTM model — improved version of SimpleRNN.
    LSTM has 3 gates:
      - Forget gate: decides what old info to throw away
      - Input gate : decides what new info to store
      - Output gate: decides what to output
    This gives it much better long-term memory than SimpleRNN.
    """
    model = keras.Sequential([
        layers.Input(shape=(time_steps, n_features)),

        # First LSTM layer — return full sequence for stacking
        layers.LSTM(64, return_sequences=True),
        layers.Dropout(0.2),

        # Second LSTM layer — return only final hidden state
        layers.LSTM(32, return_sequences=False),
        layers.Dropout(0.2),

        # Dense classifier head
        layers.Dense(16, activation='relu'),
        layers.Dense(1)   # single temperature prediction

    ], name='LSTM_Model')

    return model


lstm_model = build_lstm_model(TIME_STEPS, N_FEATURES)

lstm_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

lstm_model.summary()

---
## Step 4: Model Training & Comparison

We train for **20 epochs** and **50 epochs** to compare how training duration affects performance.

### What is an epoch?
One epoch = the model sees all training data once.
More epochs = more learning, but risk of overfitting.

In [ ]:
# ================================================================
# 4.1  TRAIN SIMPLE RNN — 20 epochs
# ================================================================

print("Training Simple RNN for 20 epochs...")
print()

history_rnn_20 = rnn_model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=64,        # process 64 sequences at a time
    validation_split=0.1, # use last 10% of training data for validation
    verbose=1,
    shuffle=False         # IMPORTANT: never shuffle time-series!
)

print("\nSimple RNN (20 epochs) training complete!")

In [ ]:
# ================================================================
# 4.2  TRAIN LSTM — 20 epochs
# ================================================================

print("Training LSTM for 20 epochs...")
print()

history_lstm_20 = lstm_model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.1,
    verbose=1,
    shuffle=False
)

print("\nLSTM (20 epochs) training complete!")

In [ ]:
# ================================================================
# 4.3  TRAIN LSTM — 50 epochs (with EarlyStopping)
# ================================================================
# We build a fresh LSTM and train for up to 50 epochs.
# EarlyStopping will stop if validation loss stops improving.

lstm_model_50 = build_lstm_model(TIME_STEPS, N_FEATURES)
lstm_model_50.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse', metrics=['mae']
)

# EarlyStopping: stop if val_loss doesn't improve for 7 consecutive epochs
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True,
    verbose=1
)

print("Training LSTM for up to 50 epochs (with EarlyStopping)...")
print()

history_lstm_50 = lstm_model_50.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1,
    shuffle=False
)

actual_epochs = len(history_lstm_50.history['loss'])
print(f"\nLSTM (50 max epochs) stopped at epoch {actual_epochs}")

In [ ]:
# ================================================================
# 4.4  COMPARE TRAINING CURVES — 20 vs 50 epochs
# ================================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Training History — RNN & LSTM\n'
             'Blue = Training | Red = Validation',
             fontsize=13, fontweight='bold')

plot_configs = [
    (history_rnn_20,  'Simple RNN — 20 epochs',  axes[0, 0]),
    (history_lstm_20, 'LSTM — 20 epochs',         axes[0, 1]),
    (history_lstm_50, f'LSTM — {actual_epochs} epochs (max 50)', axes[1, 0]),
]

for hist, title, ax in plot_configs:
    ep = range(1, len(hist.history['loss']) + 1)
    ax.plot(ep, hist.history['loss'],     'b-o', ms=3, label='Train Loss (MSE)')
    ax.plot(ep, hist.history['val_loss'], 'r-o', ms=3, label='Val Loss (MSE)')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

# Comparison: final val loss all three
model_names = ['Simple RNN\n20 epochs', 'LSTM\n20 epochs', f'LSTM\n{actual_epochs} epochs']
final_val_losses = [
    history_rnn_20.history['val_loss'][-1],
    history_lstm_20.history['val_loss'][-1],
    history_lstm_50.history['val_loss'][-1],
]
colors = ['steelblue', 'coral', 'green']
axes[1, 1].bar(model_names, final_val_losses, color=colors, edgecolor='black', width=0.5)
axes[1, 1].set_title('Final Validation Loss Comparison\n(Lower = Better)', fontweight='bold')
axes[1, 1].set_ylabel('Val MSE Loss')
axes[1, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(final_val_losses):
    axes[1, 1].text(i, v + 0.0005, f'{v:.5f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

---
## Step 5: Model Saving

In [ ]:
# ================================================================
# 5.1  SAVE TRAINED MODELS
# ================================================================
# Saving lets you reload the model later without retraining!

# Save LSTM (50 epochs) — our best model
lstm_model_50.save('rnn_time_series.h5')
print("LSTM model saved as 'rnn_time_series.h5'")

# Save Simple RNN too
rnn_model.save('simple_rnn_time_series.h5')
print("Simple RNN saved as 'simple_rnn_time_series.h5'")
print()

# Also save Keras native format (recommended for TF2)
lstm_model_50.save('rnn_time_series.keras')
print("LSTM model also saved in Keras native format: 'rnn_time_series.keras'")
print()

# Show file sizes
for fname in ['rnn_time_series.h5', 'simple_rnn_time_series.h5']:
    if os.path.exists(fname):
        size_kb = os.path.getsize(fname) / 1024
        print(f"  {fname:<35} {size_kb:.1f} KB")

print()
print("How to reload later:")
print("  loaded_model = keras.models.load_model('rnn_time_series.h5')")

---
## Step 6: Model Evaluation
### 6.1 — Compute Metrics (MAE, RMSE)

In [ ]:
# ================================================================
# 6.1  MAKE PREDICTIONS & INVERSE TRANSFORM
# ================================================================
# Step 1: Model predicts scaled values (0 to 1)
# Step 2: We must convert back to actual °C using inverse_transform

def predict_and_inverse(model, X, scaler, target_col, n_features):
    """
    Get predictions from model and convert from scaled [0,1]
    back to original temperature units (°C).
    """
    # Get model predictions (scaled)
    y_pred_scaled = model.predict(X, verbose=0).flatten()

    # To inverse transform, we need a full feature-width array
    # Create dummy array and place predictions in the target column
    dummy = np.zeros((len(y_pred_scaled), n_features))
    dummy[:, target_col] = y_pred_scaled
    # inverse_transform gives us back original-scale values
    y_pred_orig = scaler.inverse_transform(dummy)[:, target_col]

    return y_pred_orig


def actual_inverse(y_scaled, scaler, target_col, n_features):
    """Convert actual scaled targets back to °C."""
    dummy = np.zeros((len(y_scaled), n_features))
    dummy[:, target_col] = y_scaled
    return scaler.inverse_transform(dummy)[:, target_col]


# Convert actual test values back to °C
y_test_actual = actual_inverse(y_test, scaler, TARGET_COL, N_FEATURES)

# Predictions from all three models
y_pred_rnn_20  = predict_and_inverse(rnn_model,    X_test, scaler, TARGET_COL, N_FEATURES)
y_pred_lstm_20 = predict_and_inverse(lstm_model,   X_test, scaler, TARGET_COL, N_FEATURES)
y_pred_lstm_50 = predict_and_inverse(lstm_model_50, X_test, scaler, TARGET_COL, N_FEATURES)

print("Predictions generated!")
print(f"Test actual temp range : {y_test_actual.min():.1f}°C to {y_test_actual.max():.1f}°C")
print(f"LSTM-50 pred range     : {y_pred_lstm_50.min():.1f}°C to {y_pred_lstm_50.max():.1f}°C")

In [ ]:
# ================================================================
# 6.2  COMPUTE METRICS — MAE and RMSE
# ================================================================
# MAE  = Mean Absolute Error  = average |actual - predicted|
# RMSE = Root Mean Squared Error = sqrt(average (actual-predicted)²)
#        RMSE penalizes large errors more than MAE

def compute_metrics(actual, predicted, model_name):
    mae  = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mse  = mean_squared_error(actual, predicted)
    print(f"  {model_name:<25} MAE={mae:.4f}°C  RMSE={rmse:.4f}°C  MSE={mse:.6f}")
    return {'model': model_name, 'MAE': mae, 'RMSE': rmse, 'MSE': mse}


print("===== Performance Metrics (Temperature in °C) =====")
print(f"  {'Model':<25} {'MAE':>10}  {'RMSE':>10}  {'MSE':>12}")
print("  " + "-" * 60)

metrics_list = [
    compute_metrics(y_test_actual, y_pred_rnn_20,  'Simple RNN (20 ep)'),
    compute_metrics(y_test_actual, y_pred_lstm_20, 'LSTM (20 ep)'),
    compute_metrics(y_test_actual, y_pred_lstm_50, f'LSTM ({actual_epochs} ep)'),
]

print()
best = min(metrics_list, key=lambda x: x['RMSE'])
print(f"Best model by RMSE: {best['model']} with RMSE={best['RMSE']:.4f}°C")
print()
print("Interpretation:")
print("  MAE = on average, predictions are off by X degrees Celsius")
print("  RMSE = same but penalizes big errors more (more sensitive to outliers)")

### 6.3 — Visualize: Predicted vs Actual

In [ ]:
# ================================================================
# 6.3  PLOT ACTUAL vs PREDICTED TEMPERATURES
# ================================================================

N_SHOW = 500   # show first 500 test hours (~21 days)

fig, axes = plt.subplots(3, 1, figsize=(16, 12))
fig.suptitle('Actual vs Predicted Temperature (first 500 test hours)',
             fontsize=13, fontweight='bold')

for ax, y_pred, title in [
    (axes[0], y_pred_rnn_20,  'Simple RNN — 20 Epochs'),
    (axes[1], y_pred_lstm_20, 'LSTM — 20 Epochs'),
    (axes[2], y_pred_lstm_50, f'LSTM — {actual_epochs} Epochs (Best)'),
]:
    ax.plot(y_test_actual[:N_SHOW], 'b-',    linewidth=1.2, label='Actual Temperature', alpha=0.9)
    ax.plot(y_pred[:N_SHOW],        'r--',   linewidth=1.0, label='Predicted Temperature', alpha=0.8)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Time (hours)')
    ax.set_ylabel('Temperature (°C)')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print("Blue = actual temperature | Red = model prediction")
print("Closer the lines = better the model!")

In [ ]:
# ================================================================
# 6.4  SCATTER PLOT — Actual vs Predicted
# ================================================================
# A perfect model would have all points on the diagonal line

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Scatter Plot: Actual vs Predicted Temperature\n'
             'Points on diagonal = perfect prediction',
             fontsize=12, fontweight='bold')

for ax, y_pred, title in [
    (axes[0], y_pred_rnn_20,  'Simple RNN (20 ep)'),
    (axes[1], y_pred_lstm_20, 'LSTM (20 ep)'),
    (axes[2], y_pred_lstm_50, f'LSTM ({actual_epochs} ep)'),
]:
    ax.scatter(y_test_actual, y_pred, alpha=0.3, s=5, color='steelblue')
    # Perfect prediction line
    lim_min = min(y_test_actual.min(), y_pred.min()) - 1
    lim_max = max(y_test_actual.max(), y_pred.max()) + 1
    ax.plot([lim_min, lim_max], [lim_min, lim_max], 'r-', linewidth=1.5, label='Perfect')
    mae  = mean_absolute_error(y_test_actual, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred))
    ax.set_title(f'{title}\nMAE={mae:.3f}°C  RMSE={rmse:.3f}°C', fontweight='bold')
    ax.set_xlabel('Actual Temperature (°C)')
    ax.set_ylabel('Predicted Temperature (°C)')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 6.5  ERROR DISTRIBUTION PLOT
# ================================================================
# Plot the distribution of prediction errors
# Good model: errors centered at 0, narrow spread

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Prediction Error Distribution\n'
             'Good model: centered at 0, narrow spread',
             fontsize=12, fontweight='bold')

for ax, y_pred, title, color in [
    (axes[0], y_pred_rnn_20,  'Simple RNN (20 ep)',         'steelblue'),
    (axes[1], y_pred_lstm_20, 'LSTM (20 ep)',                'coral'),
    (axes[2], y_pred_lstm_50, f'LSTM ({actual_epochs} ep)', 'green'),
]:
    errors = y_test_actual - y_pred
    ax.hist(errors, bins=50, color=color, edgecolor='black', alpha=0.7)
    ax.axvline(x=0, color='red', linewidth=1.5, linestyle='--', label='Zero error')
    ax.set_title(f'{title}\nMean error={errors.mean():.3f}°C  Std={errors.std():.3f}°C',
                 fontweight='bold')
    ax.set_xlabel('Prediction Error (°C)')
    ax.set_ylabel('Count')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print("Mean error close to 0 = no systematic bias in predictions")

In [ ]:
# ================================================================
# 6.6  METRICS COMPARISON BAR CHART
# ================================================================

model_names = [m['model'] for m in metrics_list]
mae_vals    = [m['MAE']  for m in metrics_list]
rmse_vals   = [m['RMSE'] for m in metrics_list]

x     = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width/2, mae_vals,  width, label='MAE (°C)',  color='steelblue', edgecolor='black')
bars2 = ax.bar(x + width/2, rmse_vals, width, label='RMSE (°C)', color='coral',     edgecolor='black')

ax.set_title('Model Performance Comparison — MAE and RMSE\n(Lower = Better)',
             fontweight='bold', fontsize=12)
ax.set_ylabel('Error (°C)')
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.4f}', ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.4f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 6.7  ZOOM IN — 7 days of predictions (best model)
# ================================================================
# A close-up look at one week of predictions vs actual

N_WEEK = 24 * 7   # 7 days × 24 hours
start  = 100      # start from row 100 (skip first few hours of test)

plt.figure(figsize=(16, 5))
plt.plot(y_test_actual[start:start+N_WEEK],   'b-',  linewidth=1.5, label='Actual')
plt.plot(y_pred_lstm_50[start:start+N_WEEK],  'r--', linewidth=1.5, label='LSTM Predicted')

# Shade the error region
plt.fill_between(
    range(N_WEEK),
    y_test_actual[start:start+N_WEEK],
    y_pred_lstm_50[start:start+N_WEEK],
    alpha=0.2, color='orange', label='Error region'
)

plt.title(f'7-Day Close-up: Actual vs LSTM Predicted Temperature\n'
          f'(Best model: LSTM {actual_epochs} epochs)',
          fontweight='bold', fontsize=12)
plt.xlabel('Hours')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.grid(alpha=0.3)
plt.xticks(range(0, N_WEEK+1, 24), [f'Day {i+1}' for i in range(8)])
plt.tight_layout()
plt.show()
print("You can see the daily temperature cycle being predicted correctly!")

---
## Final Summary

In [ ]:
# ================================================================
# FINAL SUMMARY
# ================================================================

print("=" * 68)
print("   AIML Lab Week 11 — RNN Time-Series Forecasting — Summary")
print("=" * 68)
print()
print("DATASET:")
print(f"  Jena Climate Dataset — {len(df_hourly):,} hourly readings")
print(f"  Features : {N_FEATURES} sensor measurements")
print(f"  Target   : Temperature (°C)")
print(f"  Sequence : Past 24 hours → predict next hour")
print()
print("DATA SPLIT (chronological — no shuffling!):")
print(f"  Train : {len(X_train):,} sequences  (80%)")
print(f"  Test  : {len(X_test):,} sequences  (20%)")
print()
print("MODELS TRAINED:")
print(f"  1. Simple RNN (20 epochs)")
print(f"  2. LSTM       (20 epochs)")
print(f"  3. LSTM       ({actual_epochs} epochs, EarlyStopping, saved as rnn_time_series.h5)")
print()
print("PERFORMANCE (Test Set):")
print(f"  {'Model':<25}  {'MAE (°C)':>10}  {'RMSE (°C)':>11}")
print("  " + "-" * 50)
for m in metrics_list:
    print(f"  {m['model']:<25}  {m['MAE']:>10.4f}  {m['RMSE']:>11.4f}")
print()
print(f"BEST MODEL: {best['model']}")
print(f"  MAE = {best['MAE']:.4f}°C  (on average, off by this many degrees)")
print(f"  RMSE= {best['RMSE']:.4f}°C")
print()
print("KEY CONCLUSIONS:")
print("  1. Time-series data MUST NOT be shuffled — order matters")
print("  2. LSTM outperforms SimpleRNN — better long-term memory")
print("  3. More epochs generally reduces error but risks overfitting")
print("  4. EarlyStopping prevents wasted training and overfitting")
print("  5. Inverse-transform is essential — model outputs scaled values")
print("  6. Daily temperature cycles are clearly captured by LSTM")